# Minimal Fair-RAG Recreation (T5 Small + BM25, 5 Queries)

This notebook runs a lightweight recreation of the Fair-RAG experiment on a weak machine:
- Generator: `flanT5Small`
- Retriever: `bm25`
- Task: `LaMP 4`
- Query budget: `5`

It then publishes results by normalized EE-D intervals: `[0.0-0.2, 0.2-0.4, 0.4-0.6, 0.6-0.8, 0.8-1.0]`.

In [ ]:
from pathlib import Path
import json
import subprocess
import pandas as pd

ROOT = Path.cwd()
PY = ROOT / '.venv' / 'Scripts' / 'python.exe'
if not PY.exists():
    raise FileNotFoundError(f'Expected Python at: {PY}')

GENERATOR = 'flanT5Small'
LAMP_NUM = 4
RETRIEVER = 'bm25'
ALPHAS = [1, 2, 4, 8]

MAX_QUERIES = 5
N_SAMPLES = 5
K = 5

print('Workspace:', ROOT)
print('Python   :', PY)
print('Setup    :', {
    'generator': GENERATOR,
    'lamp_num': LAMP_NUM,
    'retriever': RETRIEVER,
    'alphas': ALPHAS,
    'max_queries': MAX_QUERIES,
    'n_samples': N_SAMPLES,
    'k': K,
})

In [ ]:
def run_cmd(args):
    print('\n> ' + ' '.join(str(x) for x in args))
    proc = subprocess.run(args, cwd=ROOT, text=True, capture_output=True)
    if proc.stdout:
        print(proc.stdout[-2500:])
    if proc.returncode != 0:
        if proc.stderr:
            print('--- stderr ---')
            print(proc.stderr[-2500:])
        raise RuntimeError(f'Command failed with exit code {proc.returncode}')
    return proc

print('Helper ready.')

## Step 1) Run Gold Baseline (alpha=8, 5 queries)
This run is needed for EU normalization in `normalize_eu.py`.

In [ ]:
run_cmd([
    str(PY), 'experiment.py',
    '--generator_name', GENERATOR,
    '--lamp_num', str(LAMP_NUM),
    '--retriever_name', 'gold',
    '--alpha', '8',
    '--k', str(K),
    '--n_samples', str(N_SAMPLES),
    '--max_queries', str(MAX_QUERIES),
    '--remove_temp_files',
])

## Step 2) Run BM25 Experiments for alpha in [1, 2, 4, 8]

In [ ]:
for alpha in ALPHAS:
    run_cmd([
        str(PY), 'experiment.py',
        '--generator_name', GENERATOR,
        '--lamp_num', str(LAMP_NUM),
        '--retriever_name', RETRIEVER,
        '--alpha', str(alpha),
        '--k', str(K),
        '--n_samples', str(N_SAMPLES),
        '--max_queries', str(MAX_QUERIES),
        '--remove_temp_files',
    ])
print('BM25 runs complete.')

## Step 3) Normalize EU (EE values are already normalized in experiment)

In [ ]:
for alpha in ALPHAS:
    run_cmd([
        str(PY), 'normalize_eu.py',
        '--generator_name', GENERATOR,
        '--lamp_num', str(LAMP_NUM),
        '--retriever_name', RETRIEVER,
        '--alpha', str(alpha),
    ])
print('EU normalization complete.')

## Step 4) Load Results and Bucket by Normalized EE-D Interval

In [ ]:
rows = []
for alpha in ALPHAS:
    fp = ROOT / 'experiment_results' / GENERATOR / f'lamp{LAMP_NUM}' / RETRIEVER / f'alpha_{alpha}_normalized.json'
    if not fp.exists():
        raise FileNotFoundError(f'Missing normalized file: {fp}')
    data = json.loads(fp.read_text())
    for qid, entry in data.items():
        rows.append({
            'alpha': alpha,
            'qid': qid,
            'ee_d': entry['EE']['disparity'],
            'ee_r': entry['EE']['relevance'],
            'eu': entry['EU']['rouge-l'],
        })

df = pd.DataFrame(rows)
bins = [0.0, 0.2, 0.4, 0.6, 0.8, 1.000001]
labels = ['0.0-0.2', '0.2-0.4', '0.4-0.6', '0.6-0.8', '0.8-1.0']
df['ee_d_interval'] = pd.cut(df['ee_d'], bins=bins, labels=labels, include_lowest=True, right=True)

print('Per-query rows loaded:', len(df))
df.head(10)

## Step 5) Publish Interval-Level Results
You get two tables:
1. interval count + average normalized EU per alpha
2. EU difference vs alpha=1 baseline inside each interval

In [ ]:
summary = (
    df.groupby(['alpha', 'ee_d_interval'], dropna=False)
      .agg(n_queries=('qid', 'nunique'), mean_eu=('eu', 'mean'), mean_ee_d=('ee_d', 'mean'))
      .reset_index()
      .sort_values(['alpha', 'ee_d_interval'])
)

pivot_eu = summary.pivot(index='ee_d_interval', columns='alpha', values='mean_eu')
if 1 in pivot_eu.columns:
    for a in ALPHAS:
        if a == 1 or a not in pivot_eu.columns:
            continue
        pivot_eu[f'eu_diff_alpha{a}_minus_alpha1'] = pivot_eu[a] - pivot_eu[1]

print('Summary by alpha + EE-D interval:')
display(summary)
print('EU and EU-difference table (interval view):')
display(pivot_eu)

In [ ]:
out_dir = ROOT / 'experiment_results' / GENERATOR / f'lamp{LAMP_NUM}' / RETRIEVER / 'notebook_5q_outputs'
out_dir.mkdir(parents=True, exist_ok=True)
summary_fp = out_dir / 'ee_d_interval_summary.csv'
pivot_fp = out_dir / 'eu_difference_by_interval.csv'

summary.to_csv(summary_fp, index=False)
pivot_eu.to_csv(pivot_fp)

print('Saved:')
print(' -', summary_fp)
print(' -', pivot_fp)